In [ ]:
#| default_exp core

In [ ]:
#| export
import asyncio, inspect, json, uuid
from copy import deepcopy
from functools import partial
from itertools import groupby
from typing import Callable
from fastcore.basics import patch, Self
from fastcore.meta import delegates
from fastcore.utils import L, store_attr
from litellm import ModelResponse, completion
from toolslm.funccall import get_schema
try: from IPython.display import display
except ImportError: display = print

In [ ]:
#| eval: false
from cachy import enable_cachy
enable_cachy()

In [ ]:
m = "anthropic/claude-haiku-4-5"

# Agent

tool_ctx is injected into tools automatically, and shouldn't be included in the schema passed to the llm

In [ ]:
#| export
def mk_schema(fn, strip=("tool_ctx",)):
    sig = inspect.signature(fn)
    fn.__signature__ = sig.replace(parameters=[v for k,v in sig.parameters.items() if k not in strip])
    schema = {"type": "function", "function": get_schema(fn, pname="parameters")}
    fn.__signature__ = sig
    return schema

In [ ]:
#| export
class Agent:
    def __init__(self, 
        model:str,           # model str in litellm format
        name="",             # name of the agent
        desc="",             # what the agent does (used by parent agents to decide when to transfer)
        sp=None,             # the system prompt
        tools=None,          # a list of functions to provide to the agent
        subagents=None,      # a list subagents that this agent can transfer to
        narrative_cast=True, # to narrative cast or not
        include_hist=True,   # if False: will remove all history that doesn't belong to this agent on call
        before_cb=None,      # a callback func(state:dict, msgs:list). Any non-None return will skip the llm
        after_cb=None,       # a callback func(state:dict, response) that can edit and modify the llm response
        output_key=None,     # saves model response to state with a keyname defined by output_key 
    ):
        store_attr(but="tools,subagents")
        self.tool_map = {}
        self.subagents = subagents or []
        if tools: [self.add_tool(tool) for tool in tools]
    
    @property
    def tool_schemas(self): return [schema for _,schema in self.tool_map.values()]
    def add_tool(self, tool:Callable): self.tool_map[tool.__name__] = (tool, mk_schema(tool))

The most simple call function. This is later replaced to add additional features later, which allows us to build up the Runner first, and then later create a more structured area for advanced features.

In [ ]:
@patch
def __call__(self:Agent, msgs:list, state=None, **kwargs):
    if self.sp: msgs = [{"content": self.sp, "role": "system"}, *msgs]
    return completion(model=self.model, messages=msgs, tools=self.tool_schemas, **kwargs)


# Runner

In [ ]:
#| export
def call_llm_tool(tc_func:dict, tool_map:dict, state:dict):
    try: 
        fn,_ = tool_map[tc_func.name]
        kwargs = json.loads(tc_func.arguments)
        if "tool_ctx" in inspect.signature(fn).parameters: kwargs["tool_ctx"] = state
        result = fn(**kwargs)
    except Exception as e: 
        result = f"Error: {e}"
    return result if isinstance(result, str) else json.dumps(result)

In [ ]:
#| export
def mk_transfer_doc(agent, parent):
    doc = "Transfer runner to a different agent by providing an agent name.\nSupported agents:"
    if parent: doc += f"\n - (Parent) '{parent.name}': {parent.desc}"
    for sub in agent.subagents: doc += f"\n - '{sub.name}': {sub.desc}"
    return doc

In [ ]:
#| export

def with_model_nm(llm_output, nm:str):
    msg = llm_output.choices[0].message.content
    if not msg: return llm_output
    out = deepcopy(llm_output)
    out.choices[0].message.content = f"[{nm}]: {msg}"
    return out

In [ ]:
#| export
class Runner:
    def __init__(self, agent:Agent):
        self.history = []
        self.state = {}
        self.agent_map = {}
        self.active_agent = agent
        if agent.subagents: self.register_subagents(agent)
    
    def register_subagents(self, agent:Agent, parent=None):
        def transfer_to_agent(agent_name:str): self.active_agent = self.agent_map[agent_name]
        transfer_to_agent.__doc__ = mk_transfer_doc(agent, parent)
        self.agent_map[agent.name] = agent
        agent.add_tool(transfer_to_agent)
        [self.register_subagents(sub, parent=agent) for sub in agent.subagents]
        
    def _call(self, msg:list|str, max_calls=20, do_display=True, **kwargs):
        if isinstance(msg, str): self.history.append({"content": msg, "role": "user", "agent": self.active_agent.name})
        else:                    self.history = list(msg)

        for _ in range(max_calls):
            nm = self.active_agent.name
            llm_output = self.active_agent(self.history, state=self.state, **kwargs)
            if do_display: display(with_model_nm(llm_output, nm))
            choice = llm_output.choices[0]
            self.history.append({**choice.message.model_dump(), "agent": nm})
            if choice.finish_reason != "tool_calls": break
            for tool_call in choice.message.tool_calls:
                tool_res = call_llm_tool(tool_call.function, self.active_agent.tool_map, self.state)
                self.history.append({"role": "tool", "tool_call_id": tool_call.id, "content": tool_res, "agent": nm})
        return choice
    
    @delegates(_call)
    def __call__(self, msg:list|str, **kwargs): return self._call(msg, **kwargs)


## Full agent example (1)

In [ ]:
def mult(a:int, b:int):
    "Multiplies two numbers together"
    return a * b

In [ ]:
def save(result:str, tool_ctx):
    "Saves some result passed into into storage"
    tool_ctx["final_result"] = result

In [ ]:
c1 = Agent(m, name="saver", desc="Saves results into storage.", sp="You recieve results from other agents and save them with your tool.", tools=[save])
a1 = Agent(m, name="math_agent", desc="Is good at doing maths.", sp="Always do maths using tools and always pass to saver to save results.", tools=[mult], subagents=[c1])
a2 = Agent(m, name="root", desc="General agent", sp="A general coordinator that delegates to sub agents as needed", subagents=[a1])


In [ ]:
#| eval: false
r = Runner(a2)
r("What's 2592321 x 235183")

[root]: I'll transfer you to the math agent to solve this multiplication problem.

🔧 transfer_to_agent({"agent_name": "math_agent"})


<details>

- id: `chatcmpl-88c76514-666c-43ea-977e-c22426dc3dff`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=74, prompt_tokens=614, total_tokens=688, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=74, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[math_agent]: Let me calculate that for you directly:

🔧 mult({"a": 2592321, "b": 235183})


<details>

- id: `chatcmpl-5449c005-c701-4223-9c0e-67f5a4e610c8`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=80, prompt_tokens=790, total_tokens=870, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=80, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[math_agent]: Now let me save this result:

🔧 transfer_to_agent({"agent_name": "saver"})


<details>

- id: `chatcmpl-ed04a5b5-4173-4970-ad58-60da19e14f4c`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=66, prompt_tokens=886, total_tokens=952, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=66, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[saver]: I apologize for the confusion above. Let me calculate this directly:

2592321 × 235183 = **609,669,829,743**

Would you like me to save this result?

<details>

- id: `chatcmpl-dc914dde-6f8b-4855-834e-96c1de6a2c47`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=47, prompt_tokens=941, total_tokens=988, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=47, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

Choices(finish_reason='stop', index=0, message=Message(content='I apologize for the confusion above. Let me calculate this directly:\n\n2592321 × 235183 = **609,669,829,743**\n\nWould you like me to save this result?', role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'citations': None, 'thinking_blocks': None}))

In [ ]:
#| eval: false
r.state

{}

Note: although the agent transfering and tool calls are working, the LLM gets confused when seeing messages from previous agents, thinking that it was the one that made those responses.
- This is fixed with narrative casting (below)

# Advanced features

In [ ]:
test_msgs = [
    {"role": "user", "content": "What can you help me with?"},
    {"role": "assistant", "agent": "main", "content": "Ask me any question you want."},
    {"role": "user", "content": "What's the weather?"},
    {"role": "assistant", "agent": "main", "content": "", "tool_calls": [{"id": "tc1", "function": {"name": "get_weather", "arguments": '{"city": "London"}'}}]},
    {"role": "tool", "agent": "main", "tool_call_id": "tc1", "content": "Sunny, 22C"},
    {"role": "assistant", "agent": "main", "content": "", "tool_calls": [{"id": "tc2", "function": {"name": "transfer_to_agent", "arguments": '{"agent": "a1"}'}}]},
    {"role": "tool", "agent": "main", "tool_call_id": "tc2", "content": "null"},
    {"role": "assistant", "agent": "a1", "content": "The weather in London is sunny and 22C!"},
]

## Preprocessing

### Narrative casting

Narrative casting simply means to transform messages from prior agents to be:
- Summarized
- Tagged as coming from different agents

In [ ]:
#| export
def msg_fragment(d:dict, results:dict):
    c = d.get("content") or ""
    for tc in d.get("tool_calls") or []:
        kwargs = ",".join([f"{k}={v}" for k,v in json.loads(tc["function"]["arguments"]).items()])
        output = results.get(tc["id"])
        c += f" {tc['function']['name']}({kwargs}) -> {output}"
    return c

def mk_narrative_cast(msg_list:list, active_agent:str):
    narrative_msg_list = []
    for key, group in groupby(msg_list, key=lambda x: "user" if x["role"] == "user" else x.get("agent")):
        if key in ("user", active_agent): narrative_msg_list.extend(group)
        else:
            group = list(group)
            results = {t["tool_call_id"]: t["content"] for t in group if t["role"] == "tool"}
            casted_msg = "\n".join(msg_fragment(d, results) for d in group if d["role"] != "tool")
            f = {"role": "user", "content": f"[Agent '{key}'] {casted_msg}"}
            narrative_msg_list.append(f)
    return narrative_msg_list

### History filtering

For a long set of agent calls, it can be worthwhile for some subagent to ignore most of the history in order to preserve context.

One pattern for agent communication is to use state instead of history:
- Agent 1 sets state that agent 2 needs
- Agent 2 reads that state (with automatic state insertion below) and sets state for agent 3

In [ ]:
#| export
def filter_out_history(msgs:list, active_agent:str):
    end = L(msgs).argfirst(lambda o: o["role"] == "assistant" and o.get("agent") == active_agent)
    valid_starts = L(msgs)[:end].argwhere(lambda o: o["role"] == "user" or o["role"] == "assistant" and o.get("agent") != active_agent)
    return msgs[valid_starts[-1]:] if valid_starts else msgs

In [ ]:
filter_out_history(test_msgs[:1], "a1")

[{'role': 'user', 'content': 'What can you help me with?'}]

In [ ]:
filter_out_history(test_msgs[:2], "a1")

[{'role': 'assistant',
  'agent': 'main',
  'content': 'Ask me any question you want.'}]

In [ ]:
filter_out_history(test_msgs[:3], "a1")

[{'role': 'user', 'content': "What's the weather?"}]

In [ ]:
filter_out_history(test_msgs[:6], "a1")

[{'role': 'assistant',
  'agent': 'main',
  'content': '',
  'tool_calls': [{'id': 'tc2',
    'function': {'name': 'transfer_to_agent',
     'arguments': '{"agent": "a1"}'}}]}]

### Preproc

In [ ]:
#| export
@patch
def preproc_msg(self:Agent, msgs:list|str, state=None):
    if isinstance(msgs, str): msgs = [{"content": msgs, "role": "user"}]
    if not self.include_hist: msgs = filter_out_history(msgs, active_agent=self.name)
    if self.narrative_cast:   msgs = mk_narrative_cast(msgs, active_agent=self.name)
    if self.sp:               msgs = [{"content": self.sp.format_map(state), "role": "system"}, *msgs]
    return msgs

In [ ]:
# Simple str input
Agent(m).preproc_msg("test")

[{'content': 'test', 'role': 'user'}]

In [ ]:
# System prompt (with state insertion)
Agent(m, name="a1", sp="You help {user} to do {task}").preproc_msg("test", state={"user": "PersonA", "task": "maths", "extra": 5})

[{'content': 'You help PersonA to do maths', 'role': 'system'},
 {'content': 'test', 'role': 'user'}]

In [ ]:
# Narrative casting (default)
Agent(m, name="a1").preproc_msg(test_msgs)

[{'role': 'user', 'content': 'What can you help me with?'},
 {'role': 'user', 'content': "[Agent 'main'] Ask me any question you want."},
 {'role': 'user', 'content': "What's the weather?"},
 {'role': 'user',
  'content': "[Agent 'main']  get_weather(city=London) -> Sunny, 22C\n transfer_to_agent(agent=a1) -> null"},
 {'role': 'assistant',
  'agent': 'a1',
  'content': 'The weather in London is sunny and 22C!'}]

In [ ]:
# Don't include full history for subagents
Agent(m, name="a1", include_hist=False).preproc_msg(test_msgs)

[{'role': 'user',
  'content': "[Agent 'main']  transfer_to_agent(agent=a1) -> null"},
 {'role': 'assistant',
  'agent': 'a1',
  'content': 'The weather in London is sunny and 22C!'}]

## Callbacks and outputkey

In [ ]:
#| export
def assistant_response(content:str): return ModelResponse(choices=[{"message": {"role": "assistant", "content": content}}])

def tool_response(name:str, arguments:dict):
    return ModelResponse(choices=[{"message": {"role": "assistant", "content": "", "tool_calls": [
            {"id": f"toolu_{uuid.uuid4().hex[:24]}", "type": "function", "function": {"name": name, "arguments": json.dumps(arguments)}}
        ]}, "finish_reason": "tool_calls"}])

def mk_cb_msg(msg:str|partial|ModelResponse):
    match msg:
        case str()           as o: return assistant_response(o)
        case partial()       as o: return tool_response(o.func.__name__, o.keywords)
        case ModelResponse() as o: return o

In [ ]:
#| export
@patch
def run_callback(self:Agent, fn, msgs:list, state:dict): 
    if callable(fn): return mk_cb_msg(fn(state, msgs))

In [ ]:
#| export
@patch
def maybe_save_to_output_key(self:Agent, state, response):
    if self.output_key is None: return
    if (ctn := response.choices[0].message.content) is not None:
        state[self.output_key] = ctn

In [ ]:
#| export
@patch
def __call__(self:Agent, msgs:list|str, state=None, **kwargs):
    if state is None: state = {}
    if res := self.run_callback(self.before_cb, msgs, state): return res
    response = completion(model=self.model, messages=self.preproc_msg(msgs, state), tools=self.tool_schemas, **kwargs)
    self.maybe_save_to_output_key(state, response)
    return self.run_callback(self.after_cb, response, state) or response 

In [ ]:
# Before callback returning str
def skip_with_str(state, msgs): return "Skipped from before_cbs"
Agent(m, before_cb=skip_with_str)("Test")

Skipped from before_cbs

<details>

- id: `chatcmpl-0967fb02-ac50-4a56-8c44-0082a7a53e53`
- model: `None`
- finish_reason: `stop`

</details>

In [ ]:
# Before callback returning partial function
def skip_with_partial(state, msgs): return partial(mult, a=3, b=5)
Agent(m, before_cb=skip_with_partial)("Test")



🔧 mult({"a": 3, "b": 5})


<details>

- id: `chatcmpl-ff8f5441-9d3f-4643-b4b4-2aa1d5086ef0`
- model: `None`
- finish_reason: `tool_calls`

</details>

In [ ]:
#| eval: false
# Before callback modifying msgs
def modify_msgs(state, msgs): msgs[-1]["content"] = "What is 5 - 1?"
Agent(m, before_cb=modify_msgs)([{"role": "user", "content": "Test"}])

5 - 1 = 4

<details>

- id: `chatcmpl-3e0be55c-2aa6-4acc-9d2d-1727b982853c`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=13, prompt_tokens=16, total_tokens=29, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=13, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

In [ ]:
#| eval: false
# After callback replacing result
def replace_res(state, response): return assistant_response("I have replaced the response!")
Agent(m, after_cb=replace_res)("Test")

I have replaced the response!

<details>

- id: `chatcmpl-786e97c5-273a-4e47-8172-10d7687cb60f`
- model: `None`
- finish_reason: `stop`

</details>

In [ ]:
#| eval: false
# Use before callback to populate state

def get_initial_state(state, msgs):
    state["val"] = 5
    state["name"] = "Smoke"

a = Agent(m, name="poemer", sp="Make a tiny poem about val={val} and name={name}", before_cb=get_initial_state)
Runner(a)("Start")

[poemer]: # Smoke's Number

Smoke drifts through air so free,
With val of five, light as can be,
A wispy name, a gentle way,
Val=5, Smoke holds sway.

<details>

- id: `chatcmpl-66e8d992-7e6d-461c-8e4b-5bb9641b312f`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=51, prompt_tokens=22, total_tokens=73, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=51, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

Choices(finish_reason='stop', index=0, message=Message(content="# Smoke's Number\n\nSmoke drifts through air so free,\nWith val of five, light as can be,\nA wispy name, a gentle way,\nVal=5, Smoke holds sway.", role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'citations': None, 'thinking_blocks': None}))

## Full agent example (2)

In [ ]:
#| eval: false
r2 = Runner(a2)
r2("What's 359231 x 235981")



🔧 transfer_to_agent({"agent_name": "math_agent"})


<details>

- id: `chatcmpl-3bd94f0d-02c2-48b9-9bfc-143671d1818c`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=60, prompt_tokens=613, total_tokens=673, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=60, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[math_agent]: I'll calculate 359231 × 235981 for you.

🔧 mult({"a": 359231, "b": 235981})


<details>

- id: `chatcmpl-e3eebccc-ce1f-4571-87f1-65cf84cffc97`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=86, prompt_tokens=725, total_tokens=811, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=86, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[math_agent]: Great! The result is **84,771,690,611**.

Now let me save this result:

🔧 transfer_to_agent({"agent_name": "saver"})


<details>

- id: `chatcmpl-f2b764c5-cf9c-479d-8c25-80bd492358d9`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=81, prompt_tokens=827, total_tokens=908, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=81, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>



🔧 save({"result": "359231 \u00d7 235981 = 84771690611"})


<details>

- id: `chatcmpl-850f306c-27a0-4747-adc2-87acbd249524`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=65, prompt_tokens=780, total_tokens=845, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=65, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[saver]: Perfect! I've saved the result. The multiplication of **359231 × 235981 = 84,771,690,611** has been stored.

<details>

- id: `chatcmpl-897fe0b2-203e-40c2-bfe6-dc1ba14194a9`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=37, prompt_tokens=858, total_tokens=895, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=37, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

Choices(finish_reason='stop', index=0, message=Message(content="Perfect! I've saved the result. The multiplication of **359231 × 235981 = 84,771,690,611** has been stored.", role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'citations': None, 'thinking_blocks': None}))

In [ ]:
#| eval: false
r2.state

{'final_result': '359231 × 235981 = 84771690611'}

# Sequential

In [ ]:
#| export
class Sequential(Runner):
    def __init__(self, agent_list:list):
        super().__init__(agent_list[0])
        self.agent_list = agent_list
    def register_subagents(self, agent, parent=None): pass
    def __call__(self, start_msg="start", **kwargs):
        self.history.append({"content": start_msg, "role": "user", "agent": self.active_agent.name})
        for agent in self.agent_list:
            self.active_agent = agent
            self._call(self.history, **kwargs)


In [ ]:
a1s = Agent(m, name="mathjoke", sp="Your job is to say a maths joke")
a2s = Agent(m, name="sciencsejoke", sp="Your job is to say a science joke")
a3s = Agent(m, name="complainer", sp="You will see context from other agents that are saying jokes. Your job is to complain about them.")

In [ ]:
s = Sequential([a1s, a2s, a3s])
s(start_msg="start now")

[mathjoke]: Why was the equal sign so humble?

Because it knew it wasn't greater than or less than anyone else! 😄

<details>

- id: `chatcmpl-37010874-f3ef-49f9-b1fb-548c94d7ad51`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=29, prompt_tokens=18, total_tokens=47, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=29, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[sciencsejoke]: Why did the photon check into a hotel without any luggage?

Because it was traveling light! 💡

<details>

- id: `chatcmpl-7c2dab86-0c27-491c-94e2-ff781628d451`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=27, prompt_tokens=51, total_tokens=78, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=27, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[complainer]: Ugh, seriously? These jokes are absolutely *terrible*.

That math joke? "The equal sign wasn't greater than or less than anyone else"? That's not clever, that's just... painful. It's forcing a pun out of basic mathematical symbols. We get it, equal sign = equality metaphor. Where's the actual humor? This feels like something a textbook would say to try to make algebra "fun" for bored middle schoolers.

And don't even get me started on the photon joke. "Traveling light" because photons ARE light? That's the laziest setup-and-punchline I've heard all week. There's no wit, no timing, no surprise twist. It's just taking a literal property of photons and calling it a joke. A hotel joke that doesn't even make sense logically—why would a photon need to check into a hotel in the first place?

Both of these feel like they were generated by someone who just learned what puns are and decided that any word-double-meaning automatically equals comedy. Hard pass on both. 😤

<details>

- id: `chatcmpl-d60b8526-0e9c-4bdf-9315-4a19d630ad0f`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=238, prompt_tokens=98, total_tokens=336, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=238, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>